# Cassini E-Ring Analysis

When publishing our Gemini results (methodology), as well as the "overall" statistics of the inference dataset, we could take the type 1 / 2 data and search for potentially interesting ice particle findings outside the E-Ring.

The E-Ring is created by the cryo-vulcanism of Enceladus. And in this approach we assume a WAS LARGER E-Ring than anticipated: 22 Rs in X-Y direction and +/- 2 RS in Z-Direction (above / below the ring-plane).

*Why is this interesting?*

The ~15 k spectra have been classified considering geometric states that are... trivial in a way: we expect water ice in the E-Ring, we expect interstellar particles from a certain direction etc. What research did not look for: water ice outside the E-Ring; due to the massive manual work of classifying spectra data!

Now in this script, before moving things into individual functions and such, we want to create a script that:

- Converts the spacecraft clock (SCLK) in a human-readable and useable format like: TDB / ET
- Computes the state vector of Cassini at an SCLK (later SCLKs of type 1 / 2 spectra) and flags the E-Ring position (inside / outside)
- Pointing of CDA: where did CDA look at? assuming e.g. a circular orbit: what are the astrodynamical properties? Is it maybe related to non-Enceladus particles?

In [1]:
!pip install spiceypy

In [2]:
import spiceypy as spice

In [3]:
import os
import re
import urllib.request
import urllib.parse
import requests
from bs4 import BeautifulSoup
import tqdm
from pathlib import Path

# URL of the Cassini SPK kernels repository
url = "https://naif.jpl.nasa.gov/pub/naif/CASSINI/kernels/spk/"
target_dir = os.path.join("kernels", "spk")

# Create target directory if it does not exist
os.makedirs(target_dir, exist_ok=True)

print(f"Fetching kernel list from {url}...")
response = requests.get(url)
response.raise_for_status()

# Parse the HTML page with BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# Pattern: 200128RU_SCPSE_XXXXX_XXXXX.bsp where XXXXX is a 5-digit number
pattern = re.compile(r"^200128RU_SCPSE_\d{5}_\d{5}\.bsp$")

matching_kernels = []
for link in soup.find_all("a"):
    href = link.get("href")
    if href:
        filename = urllib.parse.unquote(href).strip()
        if pattern.match(filename):
            matching_kernels.append(filename)

# De-duplicate the list while preserving order
matching_kernels = list(dict.fromkeys(matching_kernels))

print(f"Found {len(matching_kernels)} matching kernels:")
for k in matching_kernels:
    print(f" - {k}")


Fetching kernel list from https://naif.jpl.nasa.gov/pub/naif/CASSINI/kernels/spk/...
Found 156 matching kernels:
 - 200128RU_SCPSE_04036_04182.bsp
 - 200128RU_SCPSE_04182_04251.bsp
 - 200128RU_SCPSE_04251_04303.bsp
 - 200128RU_SCPSE_04303_05004.bsp
 - 200128RU_SCPSE_05004_05057.bsp
 - 200128RU_SCPSE_05057_05097.bsp
 - 200128RU_SCPSE_05097_05131.bsp
 - 200128RU_SCPSE_05131_05168.bsp
 - 200128RU_SCPSE_05168_05223.bsp
 - 200128RU_SCPSE_05223_05257.bsp
 - 200128RU_SCPSE_05257_05292.bsp
 - 200128RU_SCPSE_05292_05327.bsp
 - 200128RU_SCPSE_05327_06005.bsp
 - 200128RU_SCPSE_06005_06036.bsp
 - 200128RU_SCPSE_06036_06068.bsp
 - 200128RU_SCPSE_06068_06099.bsp
 - 200128RU_SCPSE_06099_06134.bsp
 - 200128RU_SCPSE_06134_06162.bsp
 - 200128RU_SCPSE_06162_06193.bsp
 - 200128RU_SCPSE_06193_06217.bsp
 - 200128RU_SCPSE_06217_06240.bsp
 - 200128RU_SCPSE_06240_06260.bsp
 - 200128RU_SCPSE_06260_06276.bsp
 - 200128RU_SCPSE_06276_06292.bsp
 - 200128RU_SCPSE_06292_06308.bsp
 - 200128RU_SCPSE_06308_06342.bsp
 - 

In [4]:
print("\nStarting downloads...")
for filename in tqdm.tqdm(matching_kernels):
    file_url = urllib.parse.urljoin(url, filename)
    dest_path = os.path.join(target_dir, filename)
    
    if os.path.exists(dest_path):
        print(f"Already exists: {filename}")
        continue
        
    with requests.get(file_url, stream=True) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                
print("All downloads complete!")


Starting downloads...


100%|██████████| 156/156 [00:00<00:00, 10230.01it/s]

Already exists: 200128RU_SCPSE_04036_04182.bsp
Already exists: 200128RU_SCPSE_04182_04251.bsp
Already exists: 200128RU_SCPSE_04251_04303.bsp
Already exists: 200128RU_SCPSE_04303_05004.bsp
Already exists: 200128RU_SCPSE_05004_05057.bsp
Already exists: 200128RU_SCPSE_05057_05097.bsp
Already exists: 200128RU_SCPSE_05097_05131.bsp
Already exists: 200128RU_SCPSE_05131_05168.bsp
Already exists: 200128RU_SCPSE_05168_05223.bsp
Already exists: 200128RU_SCPSE_05223_05257.bsp
Already exists: 200128RU_SCPSE_05257_05292.bsp
Already exists: 200128RU_SCPSE_05292_05327.bsp
Already exists: 200128RU_SCPSE_05327_06005.bsp
Already exists: 200128RU_SCPSE_06005_06036.bsp
Already exists: 200128RU_SCPSE_06036_06068.bsp
Already exists: 200128RU_SCPSE_06068_06099.bsp
Already exists: 200128RU_SCPSE_06099_06134.bsp
Already exists: 200128RU_SCPSE_06134_06162.bsp
Already exists: 200128RU_SCPSE_06162_06193.bsp
Already exists: 200128RU_SCPSE_06193_06217.bsp
Already exists: 200128RU_SCPSE_06217_06240.bsp
Already exist

In [5]:
# Download now the kernels
#    "lsk/naif0012.tls": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls",
#    "pck/pck00010.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc",
# ... into the lsk and pck directory
data_dir = Path("kernels/")

for kernel_type in ["lsk", "pck", "ck", "fk"]:
    _temp_dir = os.path.join("kernels", kernel_type)
    os.makedirs(_temp_dir, exist_ok=True)

kernels = {
    "lsk/naif0012.tls": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls",
    "pck/pck00010.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc",
    "lsk/cas00172.tsc": "https://naif.jpl.nasa.gov/pub/naif/CASSINI/kernels/sclk/cas00172.tsc",
}

# Download kernels if not already present
for name, url in kernels.items():
    path = data_dir / name
    if not path.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"{name} already exists.")

lsk/naif0012.tls already exists.
pck/pck00010.tpc already exists.
lsk/cas00172.tsc already exists.


In [6]:
# Load all existing kernels
import glob
kernel_paths = glob.glob("kernels/**/*")

for k in tqdm.tqdm(kernel_paths):
    spice.furnsh(k)

# Load also the SaturnJ2000 custom kernel
spice.furnsh("kernels/saturnj2000.tf")

100%|██████████| 159/159 [00:00<00:00, 1731.78it/s]


In [7]:
# Sample time
# 1569840352 2007-09-30T10:10:40 2007-273T10:10:40

# First we "proof" that we can convert the SCLK time to ET
sclk = 1569840352

cda_measuremnt_et = spice.scs2e(sc=-82, sclkch=str(sclk))

# And now convert it in a "human readable" format in Barcenter Dynamical Time (TDB)
cda_measuremnt_et_str = spice.timout(cda_measuremnt_et, "YYYY-MON-DD HR:MN:SC ::TDB")

print(cda_measuremnt_et_str)

2007-SEP-30 10:11:45


In [8]:
# Let's compute the state vector of Cassini w.r.t. Saturn's center (not barycenter) at the time of the measurement
cassini_IAUSaturn_state, _ = spice.spkgeo(targ=-82, et=cda_measuremnt_et, ref="IAU_SATURN", obs=699)
cassini_SaturnJ2000_state, _ = spice.spkgeo(targ=-82, et=cda_measuremnt_et, ref="SaturnJ2000", obs=699)

# Compute the speed of both state vectors
cassini_IAUSaturn_speed = spice.vnorm(cassini_IAUSaturn_state[3:])
cassini_SaturnJ2000_speed = spice.vnorm(cassini_SaturnJ2000_state[3:])

# Computeh the distance to Saturn's center
cassini_IAUSaturn_distance = spice.vnorm(cassini_IAUSaturn_state[:3])
cassini_SaturnJ2000_distance = spice.vnorm(cassini_SaturnJ2000_state[:3])

# Print the results
print(f"Cassini speed w.r.t. Saturn's center (IAU_SATURN frame): {cassini_IAUSaturn_speed:.2f} km/s")
print(f"Cassini speed w.r.t. Saturn's center (SaturnJ2000 frame): {cassini_SaturnJ2000_speed:.2f} km/s")
print(f"Cassini distance w.r.t. Saturn's center (IAU_SATURN frame): {cassini_IAUSaturn_distance:.2f} km")
print(f"Cassini distance w.r.t. Saturn's center (SaturnJ2000 frame): {cassini_SaturnJ2000_distance:.2f} km")

Cassini speed w.r.t. Saturn's center (IAU_SATURN frame): 31.69 km/s
Cassini speed w.r.t. Saturn's center (SaturnJ2000 frame): 15.62 km/s
Cassini distance w.r.t. Saturn's center (IAU_SATURN frame): 287998.75 km
Cassini distance w.r.t. Saturn's center (SaturnJ2000 frame): 287998.75 km


In [9]:
# To make things easy: We compute distances w.r.t. 1 SR (Saturn-Radius), which is based on the x / y componennt of
# Saturn's body (spheroid)
_, saturn_radii = spice.bodvrd("SATURN", "RADII", 3)
sr = saturn_radii[0]

# 1 SR
print("1 SR (Saturn Radius) in km:", sr)

1 SR (Saturn Radius) in km: 60268.0


In [11]:
# Define the "frame" of the E-Ring:
# We stay ultra conservative, to "guarantee" that we consider only measurements outside the E-Ring
# In fact, one would need to conduct astro-dynamical analysis of the orbital solution space. However, we try to keep things simple

# Distance along the x / y plane of Saturn's body (spheroid): 22 RS
# Distance above / belowz plane of Saturn's body (spheroid): 2 RS

def cassini_outside_e_ring(state_vector, sr):
    x, y, z = state_vector[:3]
    distance_xy = (x**2 + y**2)**0.5
    distance_z = abs(z)
    
    return (distance_xy > 22 * sr) or (distance_z > 2 * sr)

cassini_outside_e_ring_status = cassini_outside_e_ring(cassini_SaturnJ2000_state, sr)
print(f"Cassini outside E-Ring: {cassini_outside_e_ring_status}")

Cassini outside E-Ring: False


In [37]:
# Determine the "beginning date" and "end date" of the Cassini kernels

# Get the first and last kernel path from kernels/spk
import glob

spk_kernel_paths = sorted(glob.glob("kernels/spk/*.bsp"))
first_kernel = spk_kernel_paths[0]
last_kernel = spk_kernel_paths[-1]

kernel_paths = [first_kernel, last_kernel]

cassini_kernel_dates = []
start_dates = []
end_dates = []
for k in kernel_paths:
    # Get the Time Coverage for this specific body
    # We need a 'Window' (Double Cell) to store the result
    cover = spice.utils.support_types.SPICEDOUBLE_CELL(2000)
    spice.spkcov(k, -82, cover)

    # The coverage might consist of multiple intervals (gaps in data).
    # We usually just want the absolute start and absolute end.
    card = spice.wncard(cover) # Get number of intervals

    # Get the very first start time and very last end time
    start_et, _ = spice.wnfetd(cover, 0)         # First interval start
    _, end_et   = spice.wnfetd(cover, card - 1)  # Last interval end

    # Convert to UTC String
    start_utc = spice.timout(start_et, "YYYY-MON-DD HR:MN:SC ::TDB")
    end_utc   = spice.timout(end_et, "YYYY-MON-DD HR:MN:SC ::TDB")

    print(start_utc, end_utc)

    start_dates.append(start_et)
    end_dates.append(end_et)

ini_date = start_dates[0]
end_date = end_dates[1]

2004-FEB-06 00:00:00 2004-JUN-30 18:00:01
2017-AUG-24 00:00:00 2017-SEP-15 10:44:18


In [39]:
# Determine when Cassini entered Saturn's Sphere of Influence
saturn_soi_radius_km = 1.4321 * 1e9 * ((5.683*(10**26)) / (1.988410*(10**30)))**(2/5)
print(saturn_soi_radius_km)

# Distance between Cassini and Saturn at ini_date
cassini_initial_state, _ = spice.spkgeo(targ=-82, et=ini_date, ref="SaturnJ2000", obs=699)
cassini_initial_distance_km = spice.vnorm(cassini_initial_state[:3])
print(f"Cassini distance to Saturn at ini_date: {cassini_initial_distance_km:.2f} km")

# Determine when Cassini entered Saturn's Sphere of Influence in 1 hours steps
while cassini_initial_distance_km > saturn_soi_radius_km:
    ini_date += 3600  # Increment by 1 hour (3600 seconds)
    cassini_initial_state, _ = spice.spkgeo(targ=-82, et=ini_date, ref="SaturnJ2000", obs=699)
    cassini_initial_distance_km = spice.vnorm(cassini_initial_state[:3])
print(f"Cassini distance to Saturn at {ini_date}: {cassini_initial_distance_km:.2f} km")
print(spice.timout(ini_date, "YYYY-MON-DD HR:MN:SC ::TDB"))

54752257.67186407
Cassini distance to Saturn at ini_date: 70913778.78 km
Cassini distance to Saturn at 132328800.0: 54742586.35 km
2004-MAR-12 02:00:00


In [40]:
soi_date = ini_date

In [42]:
# Compute the percentage of time Cassini spent outside the E-Ring during its mission, with a resolution of 10 minutes
time_step = 600  # 10 minutes in seconds
current_time = soi_date
total_time = 0
outside_e_ring_time = 0
while current_time <= end_date:
    cassini_state, _ = spice.spkgeo(targ=-82, et=current_time, ref="SaturnJ2000", obs=699)
    if cassini_outside_e_ring(cassini_state, sr):
        outside_e_ring_time += time_step
    total_time += time_step
    current_time += time_step
    # Print progress every year
    if (current_time - soi_date) % (365.25 * 24 * 3600) < time_step:
        print(f"Progress: {spice.timout(current_time, 'YYYY-MON-DD HR:MN:SC ::TDB')}")
percentage_outside_e_ring = (outside_e_ring_time / total_time) * 100
print(f"Percentage of time Cassini spent outside the E-Ring: {percentage_outside_e_ring:.2f}%")

Progress: 2005-MAR-12 08:00:00
Progress: 2006-MAR-12 14:00:00
Progress: 2007-MAR-12 20:00:00
Progress: 2008-MAR-12 02:00:00
Progress: 2009-MAR-12 08:00:00
Progress: 2010-MAR-12 14:00:00
Progress: 2011-MAR-12 20:00:00
Progress: 2012-MAR-12 02:00:00
Progress: 2013-MAR-12 08:00:00
Progress: 2014-MAR-12 14:00:00
Progress: 2015-MAR-12 20:00:00
Progress: 2016-MAR-12 02:00:00
Progress: 2017-MAR-12 08:00:00
Percentage of time Cassini spent outside the E-Ring: 86.88%
